# Paper QPS-recall result

This notebook plots process-wide throughput measured with 24 concurrent query threads. It reads `results/drip/qps_curves.csv`, then replaces DRIP/HNSW series with the latest matching `.qps.csv` runs in the same directory.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'results' / 'drip').is_dir():
            return candidate
    raise RuntimeError('Could not locate results/drip from the current working directory')


REPO_ROOT = locate_repo_root()
RESULTS_DIR = REPO_ROOT / 'results'
if str(RESULTS_DIR) not in sys.path:
    sys.path.insert(0, str(RESULTS_DIR))
from latest_results import load_qps_curves

DATA_DIR = RESULTS_DIR / 'drip'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

DATASET_ORDER = ['MSMARCO', 'SIFT', 'Laion-I2I', 'Wiki']
METHOD_ORDER = ['DRIP', 'CrackIVF$^*$', 'HNSW', 'DiskANN', 'IVF4096', 'Flat']
METHOD_COLORS = {
    'DRIP': '#ec003d',
    'CrackIVF$^*$': '#008a55',
    'HNSW': '#2666e6',
    'DiskANN': '#d89000',
    'IVF4096': '#5e0fa8',
    'Flat': '#999999',
}

QPS_METHOD_ORDER = [
    'IVF4096',
    'HNSW',
    'DiskANN',
    'CrackIVF-start',
    'CrackIVF-end',
    'DRIP-start',
    'DRIP-end',
]
QPS_COLOR_ALIASES = {
    'CrackIVF-start': 'CrackIVF$^*$',
    'CrackIVF-end': 'CrackIVF$^*$',
    'DRIP-start': 'DRIP',
    'DRIP-end': 'DRIP',
}
QPS_MARKERS = {
    'IVF4096': '+',
    'HNSW': 's',
    'DiskANN': 'P',
    'CrackIVF-start': '^',
    'CrackIVF-end': 'o',
    'DRIP-start': 'D',
    'DRIP-end': '*',
}


In [ ]:
curves = load_qps_curves(DATA_DIR)

fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(18, 4))
legend_handles = {}

for column, dataset in enumerate(DATASET_ORDER):
    dataset_curves = curves[curves['dataset'] == dataset]
    axes[column].set_title(dataset, fontsize=16, pad=10)

    for method in QPS_METHOD_ORDER:
        method_curves = dataset_curves[dataset_curves['method'] == method].sort_values('recall')
        if method_curves.empty:
            continue

        color_name = QPS_COLOR_ALIASES.get(method, method)
        line = axes[column].plot(
            method_curves['recall'],
            method_curves['qps'],
            color=METHOD_COLORS[color_name],
            linestyle='--' if method.endswith('-start') else '-',
            linewidth=1.5,
            marker=QPS_MARKERS[method],
            markersize=5.5,
            markeredgewidth=1,
            markerfacecolor='none',
            alpha=0.8,
            label=method,
        )[0]
        legend_handles.setdefault(method, line)

    axes[column].set_xlim(0.80, 1.0)
    axes[column].set_yscale('log')
    axes[column].minorticks_off()
    axes[column].set_xlabel('Recall@100', fontsize=15)

axes[0].set_ylabel('QPS', fontsize=15)
fig.legend(
    [legend_handles[name] for name in QPS_METHOD_ORDER],
    QPS_METHOD_ORDER,
    loc='upper center',
    ncol=len(QPS_METHOD_ORDER),
    fontsize=12,
    frameon=True,
)
fig.tight_layout()
fig.subplots_adjust(top=0.78)

output_path = FIGURE_DIR / 'main_qps.pdf'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Saved {output_path.relative_to(REPO_ROOT)}')
plt.show()
